## **Flan-T5 avec spécialisation par fine-tuning**

# <center>**Travaux exploratoires : résumé formaté d'un texte</center>**

# <center>**III.a. Flan-T5 avec spécialisation par fine-tuning</center>**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 30/07/2025*


**Présentation du notebook :** ce notebook présente des travaux exploratoires. On explore ici la piste du modèle Flan-T5 avec **fine-tuning**.
Cette approche était plébiscitée par ChatGPT dans l'optique d'une extraction structurée de textes.

**Méthodologie :**

- le fine-tuning s'est fait sur la base d'échantillons stockant des textes et des quadruplets *(événement, lieu, moment, individus)*
- deux échantillons ont été utilisés :
  - un échantillon généré par ChatGPT : cet échantillon présente de nombreux biais (liés à d'importants défauts de représentativité) qui conduisent le modèle à halluciner, un problème bien connu des LLM
  - un échantillon *hybride*, qui s'appuie sur deux étapes.
  
    - Une première étape consiste à générer 4 échantillons intermédiaires liés à chaque composante du quadruplet *(événement, lieu, moment, individus)*. Lors de cette première étape, on vise à gérer le défaut de représentativité évoqué ci-dessus. Par exemple, pour la composante *individus*, on fait attention à désigner les individus de différentes manières possibles : nom, prénom, relation à d'autres individus (père, mère, voisin etc.), métier, ou toute autre désignation générique. Pour les prénoms, on s'est basé sur une liste très vaste de prénoms de personnes résidant en France. Idem pour les noms, et pour les autres désignations d'individus. Et de façon plus générale, on a veillé à avoir une certaine diversité dans les instances de chaque composante générique du quadruplet. L'idée est donc de générer des échantillons *représentatifs* de la population française.

    - lors de la deuxième étape, on demande à ChatGPT de générer des courts textes d'une ou deux phrases à partir de ces quadruplets. On espère ainsi avoir des textes diversifiés, afin que le modèle apprenne le mieux possible à repérer de façon abstraite les événements, les lieux, les moments et les individus dans une phrase ou un ensemble de phrases.


**Principaux résultats :**

- comme dit, l'utilisation du premier échantillon a amené le modèle à halluciner. C'est ceci qui a conduit à la conception d'un autre échantillon, plus complexe, et visant à davantage de représentativité.

- les résultats après fine-tuning sur le 2e échantillon se sont révélés assez décevants : si on arrive à capter une certaine diversité dans les quadruplets (étape 1), le passage à la génération d'un grand nombre de phrases (étape 2) s'est avérée beaucoup plus délicate. On retrouve toujours les mêmes structures génériques de phrases, du type

<center>Au moment A et au lieu B, les individus C et D se sont livrés à E</center>

<center>Selon des témoins, C s'est rendu coupable de E au lieu B au moment A</center>

- La stratégie générale de ChatGPT pour générer un grand nombre de textes semble consister à utiliser l'effet multiplicatif des croisements :

  - génération d'un nombre limité de phrases stéréotypées impliquant des éléments A, B, C, D, E,...
  - remplacer dans chacune de ces phrases stéréotypées (A, B, C, D, E,...) par des instances particulières (générées à l'étape 1)
  - par effet multiplicatif, on génère ainsi de façon simple un grand nombre de phrases.
  
- compte-tenu de cela, la stratégie déployée ici est mise en difficulté : l'ensemble des phrases ainsi constitué manque de diversité, et le modèle Flan-T5 ne pourra apprendre à extraire de manière abstraite les éléments que l'on recherche dans une phrase quelconque qui n'aurait pas été vue lors de la phase d'entraînement. En gros, notre échantillon génère de l'**overfitting**, un problème bien connu en machine learning. Le modèle surapprend et se révèle incapable de généralisation : il manque de robustesse.

**Autres approches possibles (non testées) :**

- améliorer les prompts adressés à Flan-T5 (en fait de nombreuses tentatives ont déjà été testées en vain, mais il est toujours possible de creuser)
- améliorer le passage de l'étape 1 (génération des quadruplets) à l'étape 2 (génération des phrases). Une piste serait de générer un grand nombre de petits paquets de phrases, chacun sur des thématiques propres, de façon à forcer ChatGPT (ou un autre modèle) à générer de la diversité de façon incrémentale. Il s'agit donc d'une approche hybride, dans laquelle le LLM qui génère l'échantillon est très guidé par les directives humaines. Cette approche peut être intéressante mais elle est certainement très chronophage.

**Conclusion : suite à cette expérimentation, il a finalement été décidé de changer complètement d'approche et de se tourner vers des modèles plus complexes, comme llama-70b.**

**Utilisation de la GPU en vue de l'entraînement du modèle**

Avant l'entraînement du modèle, on vérifie qu'on utilise bien la GPU pour augmenter la vitesse d'exécution de cette étape.

✅ Étapes pour activer la GPU dans Google Colab

    🔧 Clique sur le menu Exécution (ou Runtime si ton Colab est en anglais)

    ➡️ Choisis Modifier le type d'exécution (Change runtime type)

    Dans la fenêtre qui s’ouvre :

        Accélérateur matériel : choisis GPU

        (Tu peux laisser Python 3 et T4/CUDA par défaut)

    ✅ Clique sur Enregistrer

In [ ]:
import torch
torch.cuda.is_available()

True

**<u>2.a. Fine tuning du modèle flan-t5-base</u>**

**Import des librairies**

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import os

**Téléchargement du fichier de données du C: vers Google Colab**

Ce fichier est un fichier csv. C'est un échantillon qui a été généré via un prompt donné à ChatGPT.

In [ ]:
# cette ligne permet d'afficher une fenêtre lorsqu'on travaille sur Google Colab, afin d'uploader un fichier et le mettre directement là où il faut (en l'occurence, dans /content)

from google.colab import files

uploaded = files.upload() # ici, on upload le fichier "echantillon_finetuning.csv"

Saving echantillon_finetuning.csv to echantillon_finetuning.csv


In [ ]:
import pandas as pd

echantillon = pd.read_csv("echantillon_finetuning.csv")

In [ ]:
echantillon.head()

,evenement,lieu,moment,individus,texte
0,propos injurieux entendus,"85 rue de l’Égalité, Brignoles",NaN,NaN,propos injurieux entendus. Cela s'est produit ...
1,étouffement temporaire avec aliment,Centre-Val de Loire,NaN,NaN,Des faits ont été rapportés : étouffement temp...
2,menace verbale contre un enseignant,Haute-Loire,NaN,"Antoine Picard, une infirmière, un journaliste",Un événement particulier s’est produit à Haute...
3,vente illicite de médicaments,Digne-les-Bains,NaN,Moreau,"À Digne-les-Bains, un épisode notable a été ob..."
4,vol de téléphone portable dans les transports,Albi,NaN,"Julien, un menuisier, Chevalier",Un signalement a été fait concernant vol de té...


In [ ]:
echantillon["texte"][2]

"Un événement particulier s’est produit à Haute-Loire, suscitant l'attention : menace verbale contre un enseignant. Les personnes impliquées incluraient : Antoine Picard, une infirmière, un journaliste."

**Construction d'une liste de dictionnaires stockant les données d'entraînement**

In [ ]:
# import os

# os.remove("list_dict_training.json")

A partir de l'échantillon uploadé juste avant on construit une liste de dictionnaires d'entrées/sorties qui va servir de base d'entraînement pour l'étape de fine-tuning de notre modèle

In [ ]:
import pandas as pd
import json
from google.colab import files

# Création de la liste d'exemples prêts à l’emploi
liste_exemples = [
    {
        "input": row["texte"],
        "output": f"{row['evenement']}, {row['lieu']}, {row['moment']}, {row['individus']}"
    }
    for _, row in echantillon.iterrows()
]

# Sauvegarde dans un fichier JSON dans Google Colab
with open("list_dict_training.json", "w", encoding="utf-8") as f:
    json.dump(liste_exemples, f, ensure_ascii=False, indent=2)

# Téléchargement du fichier en local
files.download("list_dict_training.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# visualisation des premiers éléments
import json

for exemple in liste_exemples[:4]:
    print(json.dumps(exemple, indent=2, ensure_ascii=False))

{
  "input": "propos injurieux entendus. Cela s'est produit à 85 rue de l’Égalité, Brignoles.",
  "output": "propos injurieux entendus, 85 rue de l’Égalité, Brignoles, nan, nan"
}
{
  "input": "Des faits ont été rapportés : étouffement temporaire avec aliment. L'incident a eu lieu à Centre-Val de Loire.",
  "output": "étouffement temporaire avec aliment, Centre-Val de Loire, nan, nan"
}
{
  "input": "Un événement particulier s’est produit à Haute-Loire, suscitant l'attention : menace verbale contre un enseignant. Les personnes impliquées incluraient : Antoine Picard, une infirmière, un journaliste.",
  "output": "menace verbale contre un enseignant, Haute-Loire, nan, Antoine Picard, une infirmière, un journaliste"
}
{
  "input": "À Digne-les-Bains, un épisode notable a été observé : vente illicite de médicaments. Les personnes impliquées incluraient : Moreau.",
  "output": "vente illicite de médicaments, Digne-les-Bains, nan, Moreau"
}


**Configuration**

In [ ]:
MODEL_NAME = "google/flan-t5-base"
DATA_PATH = "list_dict_training.json"  # Ton fichier jsonl
OUTPUT_DIR = "./flan-t5-event-finetuned"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3

**Génération d'un jeu d'entraînement pour le fine-tuning**

Ces données ont été générées par ChatGPT, avec un prompt insistant sur la diversité des données d'entraînement en comparaison à un échantillon précédent qui était trop homogène.

**Prompt :**

*J'ai regardé ton échantillon, il manque de diversité :*

- *toutes les phrases sont structurées de la même façon*
- *les dates sont formatées de la même façon*
- *il y a toujours deux individus*
- *il y a toujours un 'lorsque' qui précède l'événement*

*Bref, il me faudrait un échantillon beaucoup plus hétérogène à tous les niveaux. Ce qui implique peut-être davantage d'exemples pour que le fine tuning amène à une spécialisation robuste.*

L'échantillon produit est un fichier json contenant 1 000 exemples structurés. En voici trois :

**Exemple 1 :**

{"input": "Texte : samedi en fin de journée, à 10h00, Aurélie se trouvait dans la gare de Nantes quand un coupure de courant s'est produit.", "output": "- événement : coupure de courant\n- où : la gare de Nantes, immeuble\n- quand : samedi en fin de journée, 10h00\n- qui : Aurélie"}

**Exemple 2 :**

{"input": "Texte : le 04/05 dans l'après-midi, un vol à main armée a eu lieu dans magasin. Parmi les témoins : Monsieur Dupont, Julien et Nadia.", "output": "- événement : vol à main armée\n- où : une ruelle du Vieux Lille, magasin\n- quand : le 04/05, 18h15\n- qui : Monsieur Dupont, Julien, Nadia"}

**Exemple 3 :**

{"input": "Texte : le 14/07 dans l'après-midi, un disparition a eu lieu dans parking. Parmi les témoins : Julien, Fatou et Monsieur Dupont.", "output": "- événement : disparition\n- où : le centre commercial de Marseille, parking\n- quand : le 14/07, 11h45\n- qui : Julien, Fatou, Monsieur Dupont"}

**Chargement des données d'entraînement destinées au fine-tuning du modèle**

In [ ]:
dataset = load_dataset("json", data_files={"train": DATA_PATH}, split="train")

# Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

def preprocess(example):
    model_input = tokenizer(
        example["input"], max_length=MAX_INPUT_LENGTH, truncation=True, padding="max_length"
    )
    label = tokenizer(
        example["output"], max_length=MAX_TARGET_LENGTH, truncation=True, padding="max_length"
    )
    model_input["labels"] = label["input_ids"]
    return model_input

tokenized_dataset = dataset.map(preprocess, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

**Etape de fine-tuning du modèle**

L'exécution de ce code prend un peu moins de 40 minutes avec GPU (compter 4 à 8 h avec un CPU local 4-8 coeurs)

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=10,
    save_total_limit=1,
    fp16=False,  # True si GPU avec support
    predict_with_generate=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipython-input-12-2464654264.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
10,27.815800
20,15.454900
30,6.603600
40,4.033900
50,3.463600
60,2.691900
70,1.980300
80,1.307500
90,0.804600
100,0.483200


TrainOutput(global_step=1875, training_loss=0.3540832920971016, metrics={'train_runtime': 2647.5352, 'train_samples_per_second': 5.666, 'train_steps_per_second': 0.708, 'total_flos': 1.027136028672e+16, 'train_loss': 0.3540832920971016, 'epoch': 3.0})

**Sauvegarde du modèle fine-tuné et du tokenizer**

In [ ]:
# Enregistrer le modèle et le tokenizer
model_path = "/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned"

trainer.save_model(model_path)  # Sauvegarde le modèle et le tokenizer
tokenizer.save_pretrained(model_path)

('/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned/spiece.model',
 '/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned/added_tokens.json',
 '/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned/tokenizer.json')

**<u>2.b. Utilisation du modèle fine-tuné</u>**

**Chargement du modèle flan-t5-base fine-tuné et du tokenizer**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Chemin vers ton répertoire de sauvegarde
model_path = "/content/drive/MyDrive/mon_projet/flan-t5-event-finetuned"

# Recharger le tokenizer et le modèle
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

In [ ]:
def generer_reponse(text):
    input_text = f"Texte : {text}\nRéponse :"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

texte_test = "Ce jeudi en début de soirée, Monsieur Dupont était chez son médecin, quand un homme armé s'est introduit dans le cabinet."
print(generer_reponse(texte_test))

Monsieur Dupont était chez son médecin, quand un homme armé s'est introduit dans le cabinet, nan


**Pas mal du tout sur cet exemple. On regade ce que ça donne sur les exemples de CRE simulés.**

In [ ]:
texte1 = "Du 18 au 24 juillet 2025, suite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte."
texte2 = "Une perquisition a été menée le 23 juillet à 6h00, en présence d’un officier de police judiciaire et avec l’accord du parquet."
texte3 = "Ce jeudi en début de soirée, Monsieur Dupont était chez son médecin, quand un homme armé s'est introduit dans le cabinet."

In [ ]:
print(generer_reponse(texte1))

dépassante à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles, 17 rue des Marronniers, nan, une enquête de terrain


In [ ]:
texte1

'Du 18 au 24 juillet 2025, suite à plusieurs signalements de riverains concernant des allées et venues nocturnes inhabituelles au 17 rue des Marronniers, une enquête de terrain a été ouverte.'

In [ ]:
generer_reponse(texte2)

'présence d’un officier de police judiciaire et avec l’accord du parquet, le 23 juillet à 6h00, nan'

In [ ]:
texte3 = "Les secours sont arrivés à 07h12 au bas de la rue André Malraux pour une demande d'intervention suite à un malaise d'une personne âgé de 92 ans."
generer_reponse(texte3)

"demande d'intervention suite à un malaise d'une personne âgé de 92 ans, 07h12 au bas de la rue André Malraux, nan, nan"

**<u>Bilan</u> : le modèle hallucine !**

Il invente des informations qui ne sont pas présentes dans les textes mis en input. Ce phénomène d'hallucination provient très certainement d'une **sur-représentation** de termes comme *Parc Monceau à Paris* ou *école* ou encore *Aurélie* et *Sophie* dans le jeu d'entraînement utilisé pour l'étape de fine-tuning du modèle.

Le jeu d'entraînement simulé par ChatGPT n'est pas représentatif de la population général des textes relatant des événements. Ce dédfaut de représentativité génère du **biais**.


**La réponse de ChatGPT suite à ces remarques :**

🔍 Analyse du problème
❌ Problème 1 : Hallucination

    Le modèle "invente" : "parc Monceau à Paris", "école", "Aurélie", "Sophie"…

    Cela vient probablement du fait que ces lieux/prénoms apparaissent souvent dans l’échantillon d’entraînement, même quand ils ne sont pas pertinents dans le contexte.

❌ Problème 2 : Dégradation du format

    Le format attendu (- événement :, - où :, etc.) est mal respecté.

    Tu obtiens des blocs fusionnés, voire dupliqués (- quand, - qui apparaissent plusieurs fois).

**Cet exemple est une bonne illustration de la difficulté à obtenir un jeu de données de qualité pour entraîner un modèle d'apprentissage.**

**Prochain notebook :** essayer de générer un échantillon représentatif.